In [ ]:
!pip install -q vllm qdrant-client sentence-transformers ragas langchain-google-vertexai langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/38

In [ ]:
!pip install -q langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.4 MB/s eta 0:00:00


In [ ]:
!pip install -q openai langchain-openai

In [ ]:
GCP_PROJECT = "gen-lang-client-0171046537"
from google.colab import auth
auth.authenticate_user(project_id=GCP_PROJECT)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
GCP_LOCATION    = "us-central1"
GEMINI_MODEL    = "gemini-2.5-pro"

# vLLM (generate answers)
VLLM_MODEL      = "Qwen/Qwen3-4B-Instruct-2507"
VLLM_PORT       = 8000

# Qdrant (subprocess, for rag_ms measurement)
SNAPSHOT_PATH   = "/content/drive/MyDrive/Nutrition data/nutrition_articles-final.snapshot"
COLLECTION      = "nutrition_articles"
TOP_K           = 10
QUERY_PREFIX    = "Instruct: Tim thong tin dinh duong lien quan\nQuery: "
EMBED_MODEL     = "intfloat/multilingual-e5-large-instruct"

# File paths
CONTEXTS_FILE   = "/content/drive/MyDrive/Nutrition data/eval_with_contexts_final.jsonl"
ANSWERS_FILE    = "/content/drive/MyDrive/Nutrition data/eval_answers.jsonl"
RAGAS_SUMMARY   = "/content/drive/MyDrive/Nutrition data/ragas_summary.json"
RAGAS_DETAIL    = "/content/drive/MyDrive/Nutrition data/ragas_detail.csv"
RECALL_SCREEN   = "/content/drive/MyDrive/Nutrition data/recall_screen.csv"
GAP_QUESTIONS   = "/content/drive/MyDrive/Nutrition data/gap_questions.jsonl"

RESUME = True

# Test nhanh: dat so nho (vd: 10) de check RAGAS chay dung truoc khi chay full
RAGAS_SAMPLE_LIMIT = None

CORPUS_GAP_THRESHOLD = 0.3

In [ ]:
# Cấu hình Cerebras thay cho GCP
CEREBRAS_API_KEY = ""
CEREBRAS_BASE_URL = "https://api.cerebras.ai/v1"
CEREBRAS_MODEL = "llama3.1-8b" # Hoặc model 120b tương ứng trên hệ thống Cerebras


In [ ]:
from langchain_openai import ChatOpenAI

# Khởi tạo model ChatOpenAI trỏ tới endpoint của Cerebras
cerebras_chat = ChatOpenAI(
    model=CEREBRAS_MODEL,
    openai_api_key=CEREBRAS_API_KEY,
    openai_api_base=CEREBRAS_BASE_URL,
    temperature=0
)

# Kiểm tra thử kết nối
try:
    response = cerebras_chat.invoke("Chào bạn, bạn là ai?")
    print("Kết nối Cerebras thành công!")
    print(f"Phản hồi: {response.content}")
except Exception as e:
    print(f"Lỗi kết nối: {e}")

Kết nối Cerebras thành công!
Phản hồi: Chào bạn! Mình là Qwen, một mô hình ngôn ngữ quy mô lớn được phát triển bởi Alibaba Cloud. Mình có thể giúp bạn trả lời câu hỏi, viết văn, chơi trò chơi trí tuệ, lập trình, và nhiều việc khác nữa. Bạn cần hỗ trợ điều gì hôm nay?


In [ ]:
import json
from openai import OpenAI
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

embed_model   = SentenceTransformer(EMBED_MODEL)
qdrant_client = QdrantClient(url="http://localhost:6333", timeout=60)
client        = OpenAI(base_url=f"http://localhost:{VLLM_PORT}/v1", api_key="EMPTY")

print(f"Embed model loaded: {EMBED_MODEL}")
print(f"Qdrant client connected")
print(f"vLLM client ready at port {VLLM_PORT}")

NUTRITION_SYSTEM_PROMPT = (
    "Ban la chuyen gia tu van dinh duong qua giong noi. Tuan thu:\n"
    "1. Dua vao tai lieu: Tra loi dua tren thong tin dinh duong duoc cung cap.\n"
    "2. Phong cach bac si: Bat dau bang 'Chao ban,', tu van nhu chuyen gia dinh duong.\n"
    "3. Ngan gon, de nghe: Cau tra loi se duoc doc thanh giong noi — dung cau ngan.\n"
    "4. Trung thuc: Neu khong co thong tin → noi ro 'Toi khong co thong tin ve van de nay'.\n"
    "5. Disclaimer: Ket thuc bang 'De duoc tu van chinh xac, ban nen gap bac si dinh duong.'\n"
    "/no_think"
)

def build_prompt(question: str, contexts: list) -> str:
    context_str = "\n\n".join(contexts)
    return (
        f"Tai lieu dinh duong lien quan:\n{context_str}\n"
        f"---\n"
        f"Hay tra loi DUA TREN cac tai lieu tren.\n"
        f"Cau hoi: {question}"
    )


Embed model loaded: intfloat/multilingual-e5-large-instruct
Qdrant client connected
vLLM client ready at port 8000


In [ ]:
import json
from ragas import evaluate
from ragas.metrics import AnswerCorrectness, Faithfulness, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import SingleTurnSample, EvaluationDataset
from ragas.run_config import RunConfig
from langchain_core.embeddings import Embeddings
from langchain_openai import ChatOpenAI

# Judge LLM - CEREBRAS (Increased max_tokens to prevent truncation)
llm_instance = ChatOpenAI(
    model=CEREBRAS_MODEL,
    openai_api_key=CEREBRAS_API_KEY,
    openai_api_base=CEREBRAS_BASE_URL,
    temperature=0,
    max_tokens=2048
)
judge_llm = LangchainLLMWrapper(llm_instance)

class E5Embeddings(Embeddings):
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return embed_model.encode(texts, normalize_embeddings=True).tolist()
    def embed_query(self, text: str) -> list[float]:
        return embed_model.encode([text], normalize_embeddings=True)[0].tolist()

judge_embeddings = LangchainEmbeddingsWrapper(E5Embeddings())

# Explicitly set LLM for metrics to avoid AssertionError
metrics = [AnswerCorrectness(), Faithfulness(), ContextRecall(), ContextPrecision()]
for m in metrics:
    m.llm = judge_llm
    if hasattr(m, 'embeddings'):
        m.embeddings = judge_embeddings

results = [json.loads(l) for l in open(ANSWERS_FILE, encoding="utf-8") if l.strip()]
if RAGAS_SAMPLE_LIMIT:
    results = results[:RAGAS_SAMPLE_LIMIT]
    print(f"TEST MODE: chi chay {len(results)} samples")

ragas_samples = [
    SingleTurnSample(
        user_input=r["question"],
        response=r["answer"],
        retrieved_contexts=r["contexts"],
        reference=r["reference"],
    )
    for r in results
    if r.get("answer") and r.get("contexts") and r.get("reference")
]
dataset = EvaluationDataset(samples=ragas_samples)

print("Running RAGAS with Cerebras (Fixed LLM Assignment & Max Tokens)... ")
eval_result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
    run_config=RunConfig(timeout=180, max_retries=10, max_wait=60, max_workers=2),
)
display(eval_result.to_pandas().head())

/tmp/ipykernel_633/4124232535.py:3: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import AnswerCorrectness, Faithfulness, ContextPrecision, ContextRecall
/tmp/ipykernel_633/4124232535.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import AnswerCorrectness, Faithfulness, ContextPrecision, ContextRecall
/tmp/ipykernel_633/4124232535.py:3: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import AnswerCorrectne

TEST MODE: chi chay 10 samples
RAGAS dataset: 10 samples

Running RAGAS...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[8]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[36]: TimeoutError()



=== RAGAS Scores ===
  answer_correctness        0.5036 ± 0.157  [0.313-0.784]  ██████████  (2 NaN)
  faithfulness              0.6479 ± 0.241  [0.200-0.929]  ████████████
  context_recall            0.3275 ± 0.416  [0.000-1.000]  ██████
  context_precision         0.5150 ± 0.504  [0.000-1.000]  ██████████

=== 5 MAU CONTEXT_RECALL THAP NHAT ===

  Q : Mang thai có ăn ốc được không?
  Ref: Thịt ốc có chứa nhiều đạm (11-12g protein/100g thịt ốc). Có  thể  chế  biến  ốc  thành  nhiều  món  ăn  bổ  dưỡng  tốt  cho sức khoẻ như ốc hấp lá gừng, ốc nấu chuối đậu, nem ốc... hoặc canh ốc nấu ch
  context_recall=0.000  answer_correctness=0.408
  [1] Mẹ bầu Lan Gi. mang thai 11 tuần, hay bị đầy bụng kèm táo bón, lo lắng không biết có ảnh hưởng tới thai nhi không và nên
  [2] Một số quan niệm dân gian trong thời kỳ mang thai:
Theo quan niệm dân gian phụ nữ mang thai không nên ăn rau ngót, ốc, m
  [3] Theo quan niệm dân gian phụ nữ mang thai không nên ăn rau ngót, ốc, măng hay quả đào…, nên ăn cá

In [ ]:
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

# Khởi tạo Judge LLM qua Cerebras thay vì VertexAI
# Đảm bảo bạn đã điền CEREBRAS_API_KEY ở cell phía trên
judge_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=CEREBRAS_MODEL,
        openai_api_key=CEREBRAS_API_KEY,
        openai_api_base=CEREBRAS_BASE_URL,
        temperature=0
    )
)

print(f"Judge LLM switched to Cerebras: {CEREBRAS_MODEL}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Judge LLM switched to Cerebras: qwen-3-235b-a22b-instruct-2507


/tmp/ipykernel_4086/4104352220.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(


In [ ]:
import json, pandas as pd
from ragas import evaluate
from ragas.metrics import ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas import SingleTurnSample, EvaluationDataset
from ragas.run_config import RunConfig
from langchain_openai import ChatOpenAI

# Judge LLM - CEREBRAS (Increased max_tokens to prevent truncation)
llm_instance = ChatOpenAI(
    model=CEREBRAS_MODEL,
    openai_api_key=CEREBRAS_API_KEY,
    openai_api_base=CEREBRAS_BASE_URL,
    temperature=0,
    max_tokens=2048
)
_judge_llm = LangchainLLMWrapper(llm_instance)

# Explicitly set LLM for metric
metric_recall = ContextRecall()
metric_recall.llm = _judge_llm

_results = [json.loads(l) for l in open(ANSWERS_FILE, encoding="utf-8") if l.strip()]
if RAGAS_SAMPLE_LIMIT:
    _results = _results[:RAGAS_SAMPLE_LIMIT]

_samples = [
    SingleTurnSample(
        user_input=r["question"],
        response=r["answer"],
        retrieved_contexts=r["contexts"],
        reference=r["reference"],
    )
    for r in _results
    if r.get("answer") and r.get("contexts") and r.get("reference")
]

print("Running ContextRecall screening with Cerebras (Fixed Max Tokens)... ")
_recall_result = evaluate(
    dataset=EvaluationDataset(samples=_samples),
    metrics=[metric_recall],
    llm=_judge_llm,
    run_config=RunConfig(timeout=180, max_retries=10, max_wait=60, max_workers=2),
)
display(_recall_result.to_pandas().head())

/tmp/ipykernel_4086/1577037086.py:3: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextRecall
/tmp/ipykernel_4086/1577037086.py:17: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  _judge_llm = LangchainLLMWrapper(llm_instance)


Running ContextRecall screening with Cerebras (Fixed Max Tokens)... 


Evaluating:   0%|          | 0/604 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[10]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[14]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[14]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[100]: OpenAIContextOverflowError(Error code: 400 - {'message': 'Please reduce the length of the messages or completion. Current length is 8732 while limit is 8192', 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded', 'id': ''})
ERROR:ragas.executor:Exception raised in Job[15]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[39]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[41]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.exe

KeyboardInterrupt: 